# DataQualityAgent Notebook

Часть 1: Детектив, Часть 2: Хирург, Часть 3: Аргумент.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from agents.data_quality_agent import DataQualityAgent

df = pd.read_csv('../data/raw/collected_unified.csv')
df.head()

## Часть 1: Детектив — проблемы качества

In [ ]:
agent = DataQualityAgent()
report = agent.detect_issues(df, label_col='label' if 'label' in df.columns else None)
report

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
if not missing.empty:
    plt.figure(figsize=(10,4))
    missing.plot(kind='bar')
    plt.title('Missing values by column')
    plt.tight_layout()
    plt.show()
else:
    print('No missing values')

In [ ]:
dup_count = int(df.duplicated().sum())
plt.figure(figsize=(5,3))
plt.bar(['duplicates', 'unique_rows'], [dup_count, len(df)-dup_count])
plt.title('Duplicates overview')
plt.tight_layout()
plt.show()

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
if num_cols:
    col = num_cols[0]
    plt.figure(figsize=(8,4))
    sns.boxplot(x=df[col])
    plt.title(f'Outliers check for {col}')
    plt.tight_layout()
    plt.show()
else:
    print('No numeric columns for outlier visualization')

In [ ]:
if 'label' in df.columns:
    dist = df['label'].astype(str).value_counts()
    plt.figure(figsize=(8,4))
    dist.plot(kind='bar')
    plt.title('Class imbalance')
    plt.tight_layout()
    plt.show()
else:
    print('No label column')

## Часть 2: Хирург — две стратегии чистки

In [ ]:
smart = agent.fix(df, strategy={'missing':'median','duplicates':'drop','outliers':'clip_iqr'})
aggr = agent.fix(df, strategy={'missing':'drop','duplicates':'drop','outliers':'drop'})
cmp_smart = agent.compare(df, smart)
cmp_aggr = agent.compare(df, aggr)
cmp_smart, cmp_aggr

## Часть 3: Аргумент — обоснование выбора

Для основной задачи выбирается `smart` стратегия: она снижает шум (дубли/выбросы), но сохраняет больше данных, чем `aggressive`, что обычно полезно для устойчивого обучения моделей.